In [ ]:
from pathlib import Path
import os
import sys

repo_path = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / "code/vae_convlstm/paths.py").is_file()),
    None,
)
if repo_path is None:
    raise FileNotFoundError("Open this notebook from the repository or a subfolder of it")
sys.path.insert(0, str(repo_path / "code/vae_convlstm"))
from paths import TEXAS_DATA_ROOT, TEXAS_TRAIN_DIR, TEXAS_TEST_DIR, VAE_CHECKPOINT_DIR, VAE_FIGURE_DIR
VAE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# Texas VAE latent export
Run the setup cell, then the export cell that calls `generate_latents.py`. It writes all 100 training and 20 test trajectories. The other cells are retained as exploratory analyses.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import einops
from utils import set_seed
import matplotlib.pyplot as plt
from train_utils import BatchIndicesIterator, plot_reconstruction_reference_error, load_latest_checkpoint, detect_outliers, limited_gradient
from utils import set_seed, RelativeL2Loss, format_elapsed_time, print_model_size

In [ ]:
set_seed(42)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
maeloss = torch.nn.L1Loss(reduction='sum')
mseloss = torch.nn.MSELoss(reduction='sum')


class CustomDataset(Dataset):
    def __init__(self, x_data, device):
        self.x_data = x_data
        self.x_data["y"] = torch.Tensor(self.x_data["y"])
        self.x_data["u"] = torch.Tensor(self.x_data["u"])
        self.device = device
    def __len__(self):
        return len(self.x_data["y"])

    def __getitem__(self, idx):
        return self.x_data["y"][idx].to(self.device), self.x_data["u"][idx].to(self.device)

def loss_function(x, x_hat, mean, log_var, mean_dynamics, beta=0.0, lambda_=1.0):
    reconstruction_loss = mseloss(x_hat[:,:,0], x[:,:, 0]) + lambda_ * (mseloss(x_hat[:,:, 1], x[:,:, 1]) + mseloss(x_hat[:,:, 2], x[:,:,2])) + mseloss(x_hat[:,:, 3], x[:,:,3]) + mseloss(mean[:, 1:], mean_dynamics[:, :-1])
    # KLD = - 0.5 * torch.sum(1+ log_var - mean.pow(2) - log_var.exp())

    return reconstruction_loss# + beta * KLD

def kld(mean1, log_var1, mean2, log_var2):
    return - 0.5 * torch.sum(1+ log_var1 - log_var2 - (mean1 - mean2).pow(2)/log_var2.exp() - log_var1.exp()/log_var2.exp())

In [ ]:
dataset_directory = TEXAS_TRAIN_DIR
sample_start_index = 1
sample_end_index   = 100

In [ ]:
from vae_texas import VAE_LSTM
model = VAE_LSTM(
        resolution=256, 
        in_channel=3, 
        ch_mult=(1,4,16),
        num_res_blocks=3, 
        hidden_dim=16, 
        embed_dim=8,
        latent_dim = 256
    )
model = model.to(device)

_, _ = print_model_size(model)
checkpoint_dir = VAE_CHECKPOINT_DIR
os.makedirs(checkpoint_dir, exist_ok=True)


checkpoint = torch.load(VAE_CHECKPOINT_DIR / "checkpoint_vae.pth", map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
start_epoch = checkpoint['epoch'] + 1
epoch_loss_list = checkpoint['epoch_loss_list']
frames = checkpoint.get('frames', [])
checkpoint_epoch = checkpoint['epoch']
model.eval()

# The VAE checkpoint includes its own dynamics weights.

In [ ]:
from normalization import Normalize_gaussian
rain_source = [None] * len(range(sample_start_index, sample_end_index + 1))
flow_variables = [None] * len(range(sample_start_index, sample_end_index + 1))
for i in tqdm(range(sample_start_index, sample_end_index + 1)):
    sample_directory = os.path.join(dataset_directory, f"sample_{i:05d}")
    rain_source[i - sample_start_index] = (np.load(os.path.join(sample_directory, "rain_source.npy")))
    flow_variables[i - sample_start_index] = np.load(os.path.join(sample_directory, "flow_variables.npy"))
    rain_source[i - sample_start_index] = np.repeat(rain_source[i - sample_start_index][:-1, 1], 4)
flow_variables = np.array(flow_variables).astype(np.float32)

burn_in_length = 1
flow_variables = flow_variables[:, burn_in_length:, :, :]
rain_source = np.array(rain_source).astype(np.float32)[:, :, np.newaxis]
num_time_instants = flow_variables.shape[1]

with np.load(VAE_CHECKPOINT_DIR / "mean_std.npz") as stats:
    mean, std = stats["mean"], stats["std"]

checkpoint_dir = VAE_CHECKPOINT_DIR
figure_dir = VAE_FIGURE_DIR
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok= True)
normalize_y = Normalize_gaussian(mean, std)
flow_variables = normalize_y.normalize_forw(flow_variables)

with np.load(VAE_CHECKPOINT_DIR / "mean_std_rain_source.npz") as stats:
    mean, std = stats["mean"], stats["std"]
normalize_u = Normalize_gaussian(mean, std)
rain_source = normalize_u.normalize_forw(rain_source)

coords = np.repeat(np.stack([np.repeat(2*(np.arange(0, flow_variables.shape[3], 1)/flow_variables.shape[3] - 0.5), flow_variables.shape[4]), np.tile(2*(np.arange(0, flow_variables.shape[4], 1)/flow_variables.shape[4] - 0.5), flow_variables.shape[3])], axis = 1)[None, :, :], flow_variables.shape[0], axis = 0).astype(np.float32)
data_train = {"u": rain_source, "y":flow_variables}


In [ ]:
dataset_directory = TEXAS_TEST_DIR
sample_start_index = 101
sample_end_index   = 120
burn_in_length = 1

rain_source = [None] * len(range(sample_start_index, sample_end_index + 1))
flow_variables = [None] * len(range(sample_start_index, sample_end_index + 1))
for i in tqdm(range(sample_start_index, sample_end_index + 1)):
    sample_directory = os.path.join(dataset_directory, f"sample_{i:05d}")
    rain_source[i - sample_start_index] = (np.load(os.path.join(sample_directory, "rain_source.npy")))
    flow_variables[i - sample_start_index] = np.load(os.path.join(sample_directory, "flow_variables.npy"))
    rain_source[i - sample_start_index] = np.repeat(rain_source[i - sample_start_index][:-1, 1], 4)
flow_variables = np.array(flow_variables).astype(np.float32)
burn_in_length = 1
flow_variables = flow_variables[:, burn_in_length:, :, :]
rain_source = np.array(rain_source).astype(np.float32)[:, :, np.newaxis]
flow_variables = normalize_y.normalize_forw(flow_variables)
rain_source = normalize_u.normalize_forw(rain_source)
data_valid = {"u": rain_source, "y":flow_variables}

In [ ]:
train_dataset = CustomDataset(data_train, device)
valid_dataset = CustomDataset(data_valid, device)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=1, shuffle=False)

In [ ]:
import torch
from generate_latents import load_model, load_flow_stats, generate_split

export_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
export_model = load_model(VAE_CHECKPOINT_DIR / 'checkpoint_vae.pth', export_device)
flow_mean, flow_std = load_flow_stats(VAE_CHECKPOINT_DIR / 'mean_std.npz')
export_dir = TEXAS_DATA_ROOT / 'vae_and_latents_texas'
for split in ('train', 'test'):
    generate_split(split, export_model, flow_mean, flow_std, export_device, 32, export_dir)


In [ ]:
from einops import rearrange
dataset_directory = TEXAS_TRAIN_DIR
sample_start_index = 1
sample_end_index = 100
divisions = [(1, 15)]
for j, (sample_start_index, sample_end_index) in enumerate(divisions):
    print(j)
    DEM = []
    flow_variables = []
    for i in tqdm(range(sample_start_index, sample_end_index + 1)):
        sample_directory = os.path.join(dataset_directory, f"sample_{i:05d}")
        DEM.append(np.load(os.path.join(sample_directory, "DEM.npy")))
        flow_variables.append(np.load(os.path.join(sample_directory, "flow_variables.npy"))[1::2])
        print(flow_variables[0].shape)
    DEM = np.array(DEM)
    flow_variables = np.array(flow_variables)

    burn_in_length = 1
    flow_variables = flow_variables[:, burn_in_length:, :, :]
    num_time_instants = flow_variables.shape[1]

    print('DEM shape:', DEM.shape)
    print('flow variables shape:', flow_variables.shape)
    num_all_samples = flow_variables.shape[0]
    print("Number of all samples:", num_all_samples)    
    selected_indices = np.full(num_all_samples, False, dtype=bool)
    sum_values_array = np.zeros((num_all_samples,), dtype=float)
    for i in range(num_all_samples):
        sum_value = flow_variables[i, :, 0].sum()
        sum_values_array[i] = sum_value
        if sum_value > 0:
            selected_indices[i] = True
    plt.plot(sum_values_array[selected_indices])
    plt.show()

    outlier_indices, outlier_values = detect_outliers(sum_values_array[selected_indices], method='iqr', threshold=5.0)
    true_indices = np.where(selected_indices)[0]
    selected_indices[true_indices[outlier_indices]] = False
    print('outlier indices:', outlier_indices)

    plt.plot(sum_values_array[selected_indices])
    plt.show()
    print('Number of selected samples:', np.sum(selected_indices))

    DEM = DEM[selected_indices]
    flow_variables = flow_variables[selected_indices]
    DEM = np.repeat(DEM[:, [2], None, :, :], 95, 1)/7
    print("flow_variables shape:", flow_variables.shape)
    print("DEM shape:", DEM.shape)

    # Use the first 10 samples as validation.
    X_test = {"y": flow_variables[:10], "u": DEM[:10]}
    X_train = {"y": flow_variables[10:], "u": DEM[10:]}
    print("X_train y shape:", X_train["y"].shape)  
    print("X_train u shape:", X_train["u"].shape)
    print("X_test y shape:", X_test["y"].shape)
    print("X_test u shape:", X_test["u"].shape)

    train_dataset = CustomDataset(X_train, device)
    valid_dataset = CustomDataset(X_test, device)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=1, shuffle=False)
    
    results = []
    orig_data = []
    for i, value in enumerate(train_loader):
        print(i)
        x = value
        target = x
        with torch.no_grad():
            means = []
            for j in range(target[0].shape[1]):
                means.append(model.encode(target[0][:, j]))
            mean = torch.stack(means, dim = 1)
            results.append(mean.detach().cpu().numpy())
            orig_data.append(target[0].detach().cpu().numpy())
    data = {}
    data["latent_train"] = np.concatenate(results, axis = 0)
    data["state_train"] = np.concatenate(orig_data, axis = 0)
    results = []
    orig_data = []
    for i, value in enumerate(valid_loader):
        print(i)
        x = value
        target = x
        with torch.no_grad():
            means = []
            for j in range(target[0].shape[1]):
                means.append(model.encode(target[0][:, j]))
            mean = torch.stack(means, dim = 1)
            results.append(mean.detach().cpu().numpy())
            orig_data.append(target[0].detach().cpu().numpy())
    data["latent_test"] = np.concatenate(results, axis = 0)
    data["state_test"] = np.concatenate(orig_data, axis = 0)
    save_dir = TEXAS_DATA_ROOT / "vae_and_latents_convlstm_extradata"
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, f"state_train_{j}.npy"), data["state_train"], allow_pickle = True)
    np.save(os.path.join(save_dir, f"latent_train_{j}.npy"), data["latent_train"], allow_pickle = True)
    np.save(os.path.join(save_dir, f"state_test_{j}.npy"), data["state_test"], allow_pickle = True)
    np.save(os.path.join(save_dir, f"latent_test_{j}.npy"), data["latent_test"], allow_pickle = True)

In [ ]:
def compute_relative_l2_error(prediction: np.ndarray, ground_truth: np.ndarray):
    """
    Compute the relative L2 error between predictions and labels using NumPy.

    Inputs:
    predictions: (batch_size, num_timestamps, num_features, height, width)
    labels: (batch_size, num_timestamps, num_features, height, width)

    Outputs:
    relative_error: (batch_size, num_timestamps, num_features)
    """
    prediction = np.asarray(prediction, dtype=np.float64)
    ground_truth = np.asarray(ground_truth, dtype=np.float64)

    diff_norm = np.sqrt(np.sum((prediction - ground_truth) ** 2, axis=(3, 4)))
    gt_norm = np.sqrt(np.sum(ground_truth ** 2, axis=(3, 4)))

    # gt_norm = np.where(gt_norm == 0, 1e-12, gt_norm)

    return diff_norm / gt_norm

def decode_state_sequences(latent_sequences, sequence_chunk_size=1):
    latent_sequences = np.asarray(latent_sequences, dtype=np.float32)
    latent_channels = model.post_quant_conv.in_channels
    latent_sequences = latent_sequences[:, :, :latent_channels, :, :]
    batch_size, time_steps, channels, height, width = latent_sequences.shape

    decoded_chunks = []
    for start in range(0, batch_size, sequence_chunk_size):
        chunk = latent_sequences[start:start + sequence_chunk_size]
        chunk_batch_size = chunk.shape[0]
        latent_tensor = torch.as_tensor(
            chunk.reshape(chunk_batch_size * time_steps, channels, height, width),
            device=device,
        )
        with torch.no_grad():
            decoded = model.decode(latent_tensor)
        decoded_chunks.append(
            decoded.detach().cpu().reshape(chunk_batch_size, time_steps, decoded.shape[1], decoded.shape[2], decoded.shape[3]).numpy()
        )

    return np.concatenate(decoded_chunks, axis=0)

def summarize_reconstruction_metrics(name, prediction, ground_truth, channel_names):
    prediction = np.asarray(prediction, dtype=np.float64)[:, :, :len(channel_names)]
    ground_truth = np.asarray(ground_truth, dtype=np.float64)[:, :, :len(channel_names)]

    relative_rmse_by_channel = np.mean(compute_relative_l2_error(prediction, ground_truth), axis=(0, 1))
    rmse_by_channel = np.sqrt(np.mean((prediction - ground_truth) ** 2, axis=(0, 1, 3, 4)))
    r2_by_channel = np.array([
        r2_score(ground_truth[:, :, channel].reshape(-1), prediction[:, :, channel].reshape(-1))
        for channel in range(len(channel_names))
    ])

    print(f"{name} average relative RMSE: {relative_rmse_by_channel.mean():.6f}")
    print(f"{name} average RMSE: {rmse_by_channel.mean():.6f}")
    print(f"{name} average R^2: {r2_by_channel.mean():.6f}")
    print(f"{name} relative RMSE by channel: {dict(zip(channel_names, relative_rmse_by_channel))}")
    print(f"{name} RMSE by channel: {dict(zip(channel_names, rmse_by_channel))}")
    print(f"{name} R^2 by channel: {dict(zip(channel_names, r2_by_channel))}")

channel_names = ("h", "hu", "hv")
train_predicted_states = decode_state_sequences(data["latent_train"])
train_target_states = np.asarray(data["state_train"], dtype=np.float32)[:, :, :len(channel_names)]
predicted_states = train_predicted_states
target_states = train_target_states
train_relative_l2_error = compute_relative_l2_error(train_predicted_states, train_target_states)
train_avg_relative_l2_error = np.mean(train_relative_l2_error, axis=0)
relative_l2_error = train_relative_l2_error
avg_relative_l2_error = train_avg_relative_l2_error
summarize_reconstruction_metrics("Train", train_predicted_states, train_target_states, channel_names)

In [ ]:
plt.plot(train_avg_relative_l2_error[:, 0][1:], label='h', marker='o', markersize=3)
plt.plot(train_avg_relative_l2_error[:, 1][1:], label='hu', marker='o', markersize=3)
plt.plot(train_avg_relative_l2_error[:, 2][1:], label='hv', marker='o', markersize=3)
plt.xlabel('Time Step')
plt.ylabel('Relative L2 Error')
plt.title('Relative L2 Error on train set')
plt.legend()
plt.savefig(VAE_FIGURE_DIR / "train_relative_l2_error.png")
plt.show()

In [ ]:
channel_names = ("h", "hu", "hv")
valid_predicted_states = decode_state_sequences(data["latent_test"])
valid_target_states = np.asarray(data["state_test"], dtype=np.float32)[:, :, :len(channel_names)]
predicted_states = valid_predicted_states
target_states = valid_target_states
valid_relative_l2_error = compute_relative_l2_error(valid_predicted_states, valid_target_states)
valid_avg_relative_l2_error = np.mean(valid_relative_l2_error, axis=0)
relative_l2_error = valid_relative_l2_error
avg_relative_l2_error = valid_avg_relative_l2_error
summarize_reconstruction_metrics("Validation", valid_predicted_states, valid_target_states, channel_names)

In [ ]:
# Validation metrics are computed in the previous cell.


In [ ]:
plt.plot(valid_avg_relative_l2_error[:, 0][1:], label='h', marker='o', markersize=3)
plt.plot(valid_avg_relative_l2_error[:, 1][1:], label='hu', marker='o', markersize=3)
plt.plot(valid_avg_relative_l2_error[:, 2][1:], label='hv', marker='o', markersize=3)
plt.xlabel('Time Step')
plt.ylabel('Relative L2 Error')
plt.title('Relative L2 Error on validation set')
plt.legend()
plt.savefig(VAE_FIGURE_DIR / "valid_relative_l2_error.png")
plt.show()

In [ ]:
# results = []
# orig_data = []
# for i, value in enumerate(train_loader):
#     x = value
#     target = x
#     with torch.no_grad():
#         x_hat, mean, log_var, mean_dynamics = model(target[:, :, :, :, :])
#         results.append(mean.detach().cpu().numpy())
#         orig_data.append(target.detach().cpu().numpy())
# data = {}
# data["latent_train"] = np.concatenate(results, axis = 0)
# data["state_train"] = np.concatenate(orig_data, axis = 0)
# results = []
# orig_data = []
# for i, value in enumerate(valid_loader):
#     x = value
#     target = x
#     with torch.no_grad():
#         x_hat, mean, log_var, mean_dynamics = model(target[:, :, :, :, :])
#         results.append(mean.detach().cpu().numpy())
#         orig_data.append(target.detach().cpu().numpy())
# data["latent_test"] = np.concatenate(results, axis = 0)
# data["state_test"] = np.concatenate(orig_data, axis = 0)


In [ ]:
# print(mean[0, 1:] - mean_dynamics[0, :-1])
# print(mean_dynamics[0, :-1])
# print(mean[0, 1:])
# print(mseloss(mean[:, 1:], mean_dynamics[:, :-1]))
# print(mseloss(x_hat[:,:,0], x[:,:, 0]))
# print(mseloss(x_hat[:,:, 1], x[:,:, 1]))
# print(mseloss(x_hat[:,:, 2], x[:,:,2]))
# print(mseloss(x_hat[:,:, 3], x[:,:,3]))

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for original

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for convlstm

In [ ]:
print(np.mean((dec.detach().cpu().numpy() - target.detach().cpu().numpy())**2, axis = (0, 2, 3, 4)))
#this is for convlstm finetune with residual

In [ ]:
def compute_relative_l2_error(prediction: np.ndarray, ground_truth: np.ndarray):
    """
    Compute the relative L2 error between predictions and labels using NumPy.

    Inputs:
    predictions: (batch_size, num_timestamps, num_features, height, width)
    labels: (batch_size, num_timestamps, num_features, height, width)

    Outputs:
    relative_error: (batch_size, num_timestamps, num_features)
    """
    prediction = np.asarray(prediction, dtype=np.float64)
    ground_truth = np.asarray(ground_truth, dtype=np.float64)

    diff_norm = np.sqrt(np.sum((prediction - ground_truth) ** 2, axis=(3, 4)))
    gt_norm = np.sqrt(np.sum(ground_truth ** 2, axis=(3, 4)))

    # gt_norm = np.where(gt_norm == 0, 1e-12, gt_norm)

    return diff_norm / gt_norm


import io
import imageio

colorbar_vmin = {
    'water_depth': 0.0,
    'discharge_x': -15.0,
    'discharge_y': -15.0
}
colorbar_vmax = {
    'water_depth': 40.0,
    'discharge_x': 15.0,
    'discharge_y': 15.0
}


In [ ]:
figure_dir = VAE_FIGURE_DIR

from train_utils import plot_prediction_truth_error
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())
avg_relative_l2_error = np.mean(relative_l2_error, axis=0)

frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig = plot_prediction_truth_error(dec[0, time_index, 0].detach().cpu().numpy(),
                                    target[0, time_index, 0].detach().cpu().numpy(), 
                                    title=f"Water depth at time instant {time_instant}", 
                                    vmin=colorbar_vmin['water_depth'], 
                                    vmax=colorbar_vmax['water_depth'])
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(os.path.join(figure_dir, f"reconstruction_water_depth_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 1],
#                                     target[0, time_index, 1], 
#                                     title=f"Discharge x at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_x'], 
#                                     vmax=colorbar_vmax['discharge_x'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_x_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 2],
#                                     target[0, time_index, 2], 
#                                     title=f"Discharge y at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_y'], 
#                                     vmax=colorbar_vmax['discharge_y'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_y_forecast.gif"), frames, duration=1.0, loop=0)

# # %%


In [ ]:
mean = mean.cpu().numpy()

In [ ]:
from train_utils import plot_prediction_truth_error
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())

frames = []
time_index = 0
fig = plot_prediction_truth_error(dec[0, time_index, 0].detach().cpu().numpy(),
                                target[0, time_index, 0].detach().cpu().numpy(), 
                                title=f"Water depth at time instant {time_instant}", 
                                vmin=colorbar_vmin['water_depth'], 
                                vmax=colorbar_vmax['water_depth'])
plt.savefig(VAE_FIGURE_DIR / "water_depth_time0.png")
plt.close(fig)

In [ ]:
frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))

    im00 = ax[0, 0].imshow(mean[time_index][0], cmap='viridis')
    ax[0, 0].set_title('Channel 1')
    fig.colorbar(im00, ax=ax[0, 0], pad=0.05, orientation='horizontal')

    im01 = ax[0, 1].imshow(mean[time_index][1], cmap='viridis')
    ax[0, 1].set_title('Channel 2')
    fig.colorbar(im01, ax=ax[0, 1], pad=0.05, orientation='horizontal')

    im02 = ax[0, 2].imshow(mean[time_index][2], cmap='viridis')
    ax[0, 2].set_title('Channel 3')
    fig.colorbar(im02, ax=ax[0, 2], pad=0.05, orientation='horizontal')

    im03 = ax[0, 3].imshow(mean[time_index][3], cmap='viridis')
    ax[0, 3].set_title('Channel 4')
    fig.colorbar(im03, ax=ax[0, 3], pad=0.05, orientation='horizontal')


    im10 = ax[1, 0].imshow(mean[time_index][4], cmap='viridis')
    ax[1, 0].set_title('Channel 5')
    fig.colorbar(im10, ax=ax[1, 0], pad=0.05, orientation='horizontal')

    im11 = ax[1, 1].imshow(mean[time_index][5], cmap='viridis')
    ax[1, 1].set_title('Channel 6')
    fig.colorbar(im11, ax=ax[1, 1], pad=0.05, orientation='horizontal')

    im12 = ax[1, 2].imshow(mean[time_index][6], cmap='viridis')
    ax[1, 2].set_title('Channel 7')
    fig.colorbar(im12, ax=ax[1, 2], pad=0.05, orientation='horizontal')

    im13 = ax[1, 3].imshow(mean[time_index][7], cmap='viridis')
    ax[1, 3].set_title('Channel 8')
    fig.colorbar(im13, ax=ax[1, 3], pad=0.05, orientation='horizontal')

    fig.suptitle(f"Latent variables at time instant {time_instant}")
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(VAE_FIGURE_DIR / f"latent_variables_{checkpoint_epoch}.gif", frames, duration=1.0, loop=0)

In [ ]:
figure_dir = VAE_FIGURE_DIR

from train_utils import plot_prediction_truth_error_relative
relative_l2_error = compute_relative_l2_error(dec.detach().cpu().numpy(), target.detach().cpu().numpy())
print(relative_l2_error)
frames = []
for time_instant in range(1, num_time_instants + 1):
    time_index = time_instant - 1
    fig = plot_prediction_truth_error_relative(dec[0, time_index, 0].detach().cpu().numpy(),
                                    target[0, time_index, 0].detach().cpu().numpy(), 
                                    title=f"Water depth at time instant {time_instant}", 
                                    vmin=colorbar_vmin['water_depth'], 
                                    vmax=colorbar_vmax['water_depth'])
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    frames.append(imageio.v2.imread(buf))
    buf.close()
    plt.close(fig)
imageio.mimsave(os.path.join(figure_dir, f"reconstruction_water_depth_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 1],
#                                     target[0, time_index, 1], 
#                                     title=f"Discharge x at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_x'], 
#                                     vmax=colorbar_vmax['discharge_x'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_x_forecast.gif"), frames, duration=1.0, loop=0)


# frames = []
# for time_instant in range(1, num_time_instants + 1):
#     time_index = time_instant - 1
#     fig = plot_prediction_truth_error(dec[0, time_index, 2],
#                                     target[0, time_index, 2], 
#                                     title=f"Discharge y at time instant {time_instant}", 
#                                     vmin=colorbar_vmin['discharge_y'], 
#                                     vmax=colorbar_vmax['discharge_y'])
#     buf = io.BytesIO()
#     plt.savefig(buf, format='png')
#     buf.seek(0)
#     frames.append(imageio.v2.imread(buf))
#     buf.close()
#     plt.close(fig)
# imageio.mimsave(os.path.join(figure_dir, f"reconstruction_discharge_y_forecast.gif"), frames, duration=1.0, loop=0)

# # %%
